# Prepare Western District access layers

Use GeoPandas to normalize Hong Kong open spatial datasets, work in a metric CRS for buffers/clipping, and export WGS84 GeoJSON for the web map. Download current layers from the [Hong Kong GeoData Store](https://portal.csdi.gov.hk/) or DATA.GOV.HK. The site's checked-in `static/data/access-points.geojson` is illustrative only.

Suggested inputs: CSDI / GeoInfo public open-space polygons and LCSD Outdoor Fitness Equipment point layer. Keep the original source, download date, licence, and field mapping in the notes below.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

ROOT = Path('..').resolve()
SOURCE_DIR = ROOT / 'data' / 'source'
OUTPUT = ROOT / 'static' / 'data'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Set these paths to downloaded CSDI / GeoInfo GeoJSON, GML, or Shapefile layers.
GREEN_FILE = SOURCE_DIR / 'public_green_space.geojson'
GYM_FILE = SOURCE_DIR / 'outdoor_fitness_equipment.geojson'

# Hong Kong 1980 Grid is suitable for local metric buffers and distance operations.
METRIC_CRS = 'EPSG:2326'
WEB_CRS = 'EPSG:4326'
ORIGIN_LON_LAT = (114.1285, 22.2833)  # approximate placeholder; survey/verify entrance

## Read and inspect sources

Check field names and coordinate systems before proceeding. Public-space polygons should remain polygons so the clip step identifies the portion intersecting the walking buffer; fitness equipment is commonly represented as point locations.

In [ ]:
def read_layer(path: Path) -> gpd.GeoDataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing downloaded source layer: {path}')
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f'{path.name} has no CRS; assign its documented source CRS before reprojection')
    return layer

green = read_layer(GREEN_FILE)
fitness = read_layer(GYM_FILE)
print('Green CRS / rows:', green.crs, len(green))
print('Fitness CRS / rows:', fitness.crs, len(fitness))
display(green.head(3))
display(fitness.head(3))

## Create a walking-area clip with the app contour

Run the FastAPI app with a Google Maps Platform key, call `POST /api/isochrone` with the same origin, walking-time and maximum-slope settings, then save its JSON response as `data/source/walking_isochrone.json`. The endpoint uses Google walking-route matrix samples and slope-screened edge routes; it does not create a straight-line circle. Google API usage may be billable. The site preview uses fictional demo routes and cannot produce this contour.

The next cell reads the returned GeoJSON Feature (either saved by itself or inside the endpoint response), projects it into EPSG:2326, then clips source green-space polygons and fitness points to the walking contour.


In [ ]:
import json

ISOCHRONE_FILE = SOURCE_DIR / 'walking_isochrone.json'
if not ISOCHRONE_FILE.exists():
    raise FileNotFoundError('Save the Google /api/isochrone response to ' + str(ISOCHRONE_FILE))
document = json.loads(ISOCHRONE_FILE.read_text())
feature = document.get('feature') or (document if document.get('type') == 'Feature' else None)
if feature is None or not feature.get('geometry'):
    raise ValueError('The saved response contains no usable walking-area polygon. Check the API message and sample count.')
reach = gpd.GeoDataFrame.from_features([feature], crs=WEB_CRS).to_crs(METRIC_CRS)
green_m = green.to_crs(METRIC_CRS)
fitness_m = fitness.to_crs(METRIC_CRS)
green_clip = gpd.clip(green_m, reach)
fitness_clip = gpd.clip(fitness_m, reach)
print('Walking contour method:', feature.get('properties', {}).get('method', 'source metadata not supplied'))
print(f'{len(green_clip)} green polygons and {len(fitness_clip)} fitness points within the sampled contour')

## Normalize features and export web layers

Update `name_column` and optional identifier fields to match the source layer's documented schema. Do not discard source identifiers or metadata needed to trace records back to the publisher.

In [ ]:
def normalize(gdf: gpd.GeoDataFrame, kind: str, name_column: str) -> gpd.GeoDataFrame:
    out = gdf.to_crs(WEB_CRS).copy()
    out['kind'] = kind
    out['name'] = out[name_column].fillna(kind.title()) if name_column in out else kind.title()
    out['source'] = 'Hong Kong GeoData Store / source layer metadata'
    return out

# Replace these field names after inspecting the downloaded layers.
green_web = normalize(green_clip, 'green', 'name_en')
fitness_web = normalize(fitness_clip, 'gym', 'facility_name_en')
combined = gpd.GeoDataFrame(pd.concat([green_web, fitness_web], ignore_index=True), crs=WEB_CRS)
combined.to_file(OUTPUT / 'access-points.geojson', driver='GeoJSON', index=False)
print('Wrote', OUTPUT / 'access-points.geojson', 'features:', len(combined))

## Notes for interpreting the map

The static preview is a functioning OpenStreetMap/OpenFreeMap 3D city view with illustrative destination routes. It is not connected to Google Maps Platform, and its demo route and elevation values are not survey data.

With a configured backend, the app obtains Google pedestrian route times and geometry, samples Google elevation along each route, and estimates each segment angle as `atan(abs(Δelevation) / horizontal segment distance)`. Its 24-direction contour is a sampled approximation, not a complete pedestrian-network isochrone; it may miss route alternatives, alleys, stairs, kerbs, crossing quality, surface condition, closures, or step-free entrances. Elevation resolution can miss short ramps or steps. Field checks and a verified step-free pedestrian network are required before product or policy decisions.

For reproducibility, record source URLs, retrieval date, licence, source CRS, name/ID field mappings, and filters applied when preparing each layer. Preserve the original source metadata.